# Project 7 — CNN From Scratch (CIFAR-10)
```
input(32x32x3) -> [Conv3x3 -> BN -> ReLU -> MaxPool]x2 -> Flatten -> Linear -> 10 logits
```
**Goal:** feel why convolution beats a flat MLP (local weights, sharing, translation invariance).
**Objectives:** (1) hand-write `conv2d` and match torch; (2) train a tiny CNN on CPU;
(3) compute the receptive field by hand; (4) ablate BatchNorm; (5) time k3 vs k5.

In [ ]:
import os # need os to create results/ and check the working directory
import random # need random to seed the stdlib RNG
import time # need time to time epochs and micro-benchmarks
import csv # need csv to write metrics tables portably
import numpy as np # need numpy for data handling and the from-scratch conv
import torch # need torch for the CNN, training loop, and reference conv
import torch.nn as nn # need nn for Conv2d/BatchNorm2d/Linear modules
import torch.nn.functional as F # need F for relu/max_pool2d/cross_entropy calls
import matplotlib # need matplotlib for loss/accuracy plots
matplotlib.use('Agg') # force non-interactive backend so headless execute works
import matplotlib.pyplot as plt # need pyplot for the curves figure
SEED = 0 # fixed seed so every run is reproducible
random.seed(SEED) # seed the stdlib RNG
np.random.seed(SEED) # seed the numpy RNG
torch.manual_seed(SEED) # seed the torch RNG
torch.set_num_threads(4) # cap CPU threads so timings are stable
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # use GPU if present, else CPU
print('device:', DEVICE) # log the device for the report
if DEVICE.type == 'cuda': # only print GPU name when we really have one
    print(torch.cuda.get_device_name(0)) # log GPU model for the benchmark table
os.makedirs('./results', exist_ok=True) # create results/ for CSVs and plots

In [ ]:
IMG = 32 # CIFAR images are 32x32 pixels
NCLASS = 10 # CIFAR-10 has 10 classes
NTRAIN = 4000 # subset size: trains in ~1 min on CPU, still learns
NTEST = 1000 # held-out size: +-1.5pp noise on accuracy, good enough
BATCH = 128 # minibatch size balancing gradient noise vs throughput
def make_synthetic(n, gen): # build fake CIFAR-shaped data if download fails
    X = gen.random((n, 3, IMG, IMG), dtype=np.float32) # uniform noise images in [0,1)
    y = gen.integers(0, NCLASS, size=n).astype(np.int64) # random labels 0..9
    for i in range(n): # stamp a class-colored patch so learnable signal exists
        c = int(y[i]) # class index drives patch color and position
        X[i, c % 3, (c*2):(c*2+8), (c*3):(c*3+8)] += 1.0 # bright 8x8 patch encodes the label
    return np.clip(X, 0, 1), y # clip back to the valid pixel range
try: # try real CIFAR-10 first (one-time ~170MB download, then cached)
    from torchvision import datasets # need torchvision datasets for CIFAR-10
    import torchvision.transforms as T # need transforms to convert PIL to tensor
    tf = T.Compose([T.ToTensor()]) # map PIL 0..255 to float tensor 0..1
    tr_full = datasets.CIFAR10(root='./data', train=True, download=True, transform=tf) # fetch train split
    te_full = datasets.CIFAR10(root='./data', train=False, download=True, transform=tf) # fetch test split
    pick = np.random.default_rng(SEED) # seeded RNG for reproducible subsetting
    tri = pick.choice(len(tr_full), NTRAIN, replace=False) # random train indices
    tei = pick.choice(len(te_full), NTEST, replace=False) # random test indices
    Xtr = torch.stack([tr_full[int(i)][0] for i in tri]).numpy() # gather train images
    ytr = np.array([tr_full[int(i)][1] for i in tri]) # gather train labels
    Xte = torch.stack([te_full[int(i)][0] for i in tei]).numpy() # gather test images
    yte = np.array([te_full[int(i)][1] for i in tei]) # gather test labels
    SRC = 'cifar10-subset' # record that we trained on real data
except Exception as e: # offline machine or download blocked -> synthetic fallback
    print('CIFAR download failed, using synthetic:', e) # log why we fell back
    gen = np.random.default_rng(SEED) # seeded generator for the fake data
    Xtr, ytr = make_synthetic(NTRAIN, gen) # fake train set with planted signal
    Xte, yte = make_synthetic(NTEST, gen) # fake test set from the same stream
    SRC = 'synthetic' # record the fallback in all outputs
print('source:', SRC, Xtr.shape, ytr.shape) # log source and array shapes
MU = Xtr.mean(axis=(0, 2, 3), keepdims=True) # per-channel mean over the train set
SD = Xtr.std(axis=(0, 2, 3), keepdims=True) + 1e-6 # per-channel std (+eps avoids div-by-zero)
Xtr = ((Xtr - MU) / SD).astype(np.float32) # standardize train images (zero mean, unit variance)
Xte = ((Xte - MU) / SD).astype(np.float32) # standardize test with TRAIN stats (no leakage)
tr_ds = torch.utils.data.TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(ytr)) # wrap train arrays
te_ds = torch.utils.data.TensorDataset(torch.from_numpy(Xte), torch.from_numpy(yte)) # wrap test arrays
tr_ld = torch.utils.data.DataLoader(tr_ds, batch_size=BATCH, shuffle=True, num_workers=0) # shuffled train loader
te_ld = torch.utils.data.DataLoader(te_ds, batch_size=512, shuffle=False, num_workers=0) # plain test loader
print('train batches:', len(tr_ld)) # sanity check on loader length

In [ ]:
def conv2d_naive(x, w, b=None, stride=1, pad=0): # cross-correlation on one image, no libraries
    C, H, Wd = x.shape # unpack input channels, height, width
    F, _, KH, KW = w.shape # unpack filter count and kernel height/width
    xq = np.pad(x, ((0, 0), (pad, pad), (pad, pad))) # zero-pad H/W so borders are covered
    OH = (H + 2*pad - KH)//stride + 1 # output height from the conv-size formula
    OW = (Wd + 2*pad - KW)//stride + 1 # output width from the conv-size formula
    out = np.zeros((F, OH, OW), dtype=np.float32) # allocate the output feature maps
    for f in range(F): # loop over filters (slow but explicit is the point)
        for i in range(OH): # loop over output rows
            for j in range(OW): # loop over output columns
                patch = xq[:, i*stride:i*stride+KH, j*stride:j*stride+KW] # slice this output's receptive patch
                out[f, i, j] = float((patch * w[f]).sum()) # dot the filter against the patch
        if b is not None: # optional bias term
            out[f] += b[f] # broadcast one scalar bias over the whole map
    return out # return (F, OH, OW) activations
tg = np.random.default_rng(1) # separate fixed RNG so the self-check is deterministic
tx = tg.random((2, 5, 5), dtype=np.float32) # tiny 2-channel 5x5 input
tw = tg.random((3, 2, 3, 3), dtype=np.float32) # 3 filters of size 3x3
tb = tg.random(3, dtype=np.float32) # one bias per filter
mine = conv2d_naive(tx, tw, tb, stride=1, pad=1) # run our implementation
ref = F.conv2d(torch.from_numpy(tx[None]), torch.from_numpy(tw), torch.from_numpy(tb), padding=1).numpy()[0] # torch reference
print('conv max-abs-diff:', float(np.abs(mine - ref).max())) # must be ~1e-6 or our math is wrong
assert float(np.abs(mine - ref).max()) < 1e-4 # fail loudly if the hand conv disagrees
class TinyCNN(nn.Module): # 2 conv blocks + linear head, ~60k params
    def __init__(self, use_bn=True): # flag lets the ablation switch BatchNorm off
        super().__init__() # required nn.Module initialization
        self.use_bn = use_bn # store the ablation switch
        self.c1 = nn.Conv2d(3, 32, 3, padding=1) # conv1: 3->32 channels, keeps 32x32
        self.b1 = nn.BatchNorm2d(32) # norm1: stabilizes conv1 output scale
        self.c2 = nn.Conv2d(32, 64, 3, padding=1) # conv2: 32->64 channels, keeps 16x16 post-pool
        self.b2 = nn.BatchNorm2d(64) # norm2: stabilizes conv2 output scale
        self.fc = nn.Linear(64*8*8, NCLASS) # head: flattened 8x8 maps -> 10 logits
    def forward(self, x): # conv->norm->relu->pool twice, then flatten + linear
        x = self.c1(x) # first convolution: learn 32 edge/color filters
        x = self.b1(x) if self.use_bn else x # optionally normalize (ablation point)
        x = F.relu(x) # nonlinearity so the net learns more than linear maps
        x = F.max_pool2d(x, 2) # downsample 32->16: max keeps features, adds shift invariance
        x = self.c2(x) # second convolution on the 16x16 maps
        x = self.b2(x) if self.use_bn else x # optionally normalize (ablation point)
        x = F.relu(x) # second nonlinearity
        x = F.max_pool2d(x, 2) # downsample 16->8 before the classifier
        x = x.flatten(1) # flatten maps to a vector, keep the batch dim
        return self.fc(x) # linear head outputs one logit per class
net = TinyCNN().to(DEVICE) # instantiate the model on the selected device
print('params:', sum(p.numel() for p in net.parameters())) # log total parameter count

In [ ]:
def run_epoch(model, loader, opt, train): # one epoch shared by train and eval paths
    model.train(train) # train mode enables BN updates; eval mode freezes them
    loss_sum, correct, n = 0.0, 0, 0 # accumulators for mean loss and accuracy
    for xb, yb in loader: # iterate over minibatches
        xb, yb = xb.to(DEVICE), yb.to(DEVICE) # move the batch to the device
        out = model(xb) # forward pass to class logits
        loss = F.cross_entropy(out, yb) # softmax cross-entropy, averaged over batch
        if train: # only backpropagate when training
            opt.zero_grad() # clear stale gradients from the previous step
            loss.backward() # backprop: dL/dparams via the chain rule
            opt.step() # optimizer step updates every weight
        bs = xb.size(0) # batch size weights the running average
        loss_sum += float(loss.detach()) * bs # accumulate size-weighted batch loss (detach avoids grad warning)
        correct += int((out.argmax(1) == yb).sum()) # count correct predictions
        n += bs # count samples seen this epoch
    return loss_sum / n, correct / n # return mean loss and accuracy
EPOCHS = 5 # epoch budget: learns on CPU in ~1 min, enough to beat chance clearly
opt = torch.optim.Adam(net.parameters(), lr=3e-3) # Adam adapts per-param step sizes
hist = {'loss': [], 'acc': [], 't': []} # history lists feed the CSV and the plots
for ep in range(EPOCHS): # main training loop
    t0 = time.time() # start the epoch timer
    l, a = run_epoch(net, tr_ld, opt, True) # train one full epoch
    dt = time.time() - t0 # stop the epoch timer
    hist['loss'].append(l) # record train loss
    hist['acc'].append(a) # record train accuracy
    hist['t'].append(dt) # record epoch wall time
    print(f'ep{ep+1}/{EPOCHS} loss={l:.4f} acc={a:.4f} {dt:.1f}s') # progress log per epoch

In [ ]:
te_loss, te_acc = run_epoch(net, te_ld, None, False) # evaluate once on held-out test data
print(f'test loss={te_loss:.4f} acc={te_acc:.4f} ({SRC})') # headline metric plus data source
with open('./results/metrics.csv', 'w', newline='') as f: # open the metrics CSV for writing
    w = csv.writer(f) # need a writer for row output
    w.writerow(['epoch', 'train_loss', 'train_acc', 'secs']) # header row
    for i in range(EPOCHS): # one row per training epoch
        w.writerow([i+1, f"{hist['loss'][i]:.4f}", f"{hist['acc'][i]:.4f}", f"{hist['t'][i]:.2f}"]) # write epoch stats
    w.writerow(['test', f'{te_loss:.4f}', f'{te_acc:.4f}', '']) # final row holds test metrics
fig, ax = plt.subplots(1, 2, figsize=(10, 4)) # two panels: loss curve and accuracy curve
ax[0].plot(hist['loss'], marker='o') # train loss vs epoch
ax[0].set_title('train loss') # left panel title
ax[0].set_xlabel('epoch') # left x label
ax[1].plot(hist['acc'], marker='o') # train accuracy vs epoch
ax[1].set_title('train acc') # right panel title
ax[1].set_xlabel('epoch') # right x label
fig.suptitle(f'P7 TinyCNN ({SRC}) test_acc={te_acc:.3f}') # figure title carries the headline number
fig.savefig('./results/curves.png', dpi=100) # save the plot for the report
plt.close(fig) # free the figure memory

In [ ]:
rf, stride = 1, 1 # start: each activation sees exactly 1 input pixel
for k, s in [(3, 1), (2, 2), (3, 1), (2, 2)]: # (kernel,stride) of conv,pool,conv,pool
    rf = rf + (k-1)*stride # receptive field grows by (k-1) input-pixels per layer
    stride *= s # track cumulative stride for the next layer's contribution
print('receptive field at classifier:', rf, 'x', rf, 'of 32x32 input') # 10x10: local, not global
x = torch.randn(128, 3, IMG, IMG, device=DEVICE) # fixed probe batch for fair timing
for kw, tag in [(3, 'k3'), (5, 'k5')]: # compare stacked-3x3 style vs a bigger first kernel
    m = nn.Conv2d(3, 32, kw, padding=kw//2).to(DEVICE) # same in/out shape, different kernel
    t0 = time.time() # start the micro-benchmark timer
    with torch.no_grad(): # no gradients: pure forward-pass timing
        for _ in range(20): # repeat to smooth out timer noise
            m(x) # forward pass
    if DEVICE.type == 'cuda': # GPU kernels run async
        torch.cuda.synchronize() # wait so the timing is real
    ms = (time.time()-t0)*1000 # total milliseconds for 20 forwards
    print(f'{tag}: params={m.weight.numel()}, 20x fwd={ms:.1f}ms') # params vs speed tradeoff
net_nb = TinyCNN(use_bn=False).to(DEVICE) # identical net with BatchNorm disabled
opt_nb = torch.optim.Adam(net_nb.parameters(), lr=3e-3) # fresh optimizer for the fresh net
for ep in range(EPOCHS): # same epoch budget so the comparison is fair
    l_nb, a_nb = run_epoch(net_nb, tr_ld, opt_nb, True) # train one epoch without BN
    print(f'no-bn ep{ep+1}/{EPOCHS} loss={l_nb:.4f} acc={a_nb:.4f}') # progress log
lte_nb, ate_nb = run_epoch(net_nb, te_ld, None, False) # evaluate the no-BN model on test
print(f'ablation test_acc with-BN={te_acc:.4f} without-BN={ate_nb:.4f}') # headline comparison
with open('./results/ablation.csv', 'w', newline='') as f: # save the ablation table
    w = csv.writer(f) # need a writer for row output
    w.writerow(['variant', 'test_loss', 'test_acc']) # header row
    w.writerow(['with_bn', f'{te_loss:.4f}', f'{te_acc:.4f}']) # main model row
    w.writerow(['without_bn', f'{lte_nb:.4f}', f'{ate_nb:.4f}']) # no-BN row, same budget
print('done: see ./results/metrics.csv, ablation.csv, curves.png') # point at all artifacts

## What to write in your README (7 questions)
1. What did I build? 2. How does convolution work (sharing + pooling)?
3. What test accuracy / timing? 4. What surprised you? 5. Bottleneck?
6. What if no-BN / k5 / more epochs? 7. What next (deeper net, augmentation)?